## Personal Notebook, SVMs (Support Vector Machines), Lotta Kauppinen

##### Using the cleaned_100k.csv and the same skeleton as in the assignment notebook.

### Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from imblearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay
)

from imblearn.over_sampling import SMOTE
from sklearn.svm import SVC

### Reading the CSV and split 60/20/20

In [2]:
df = pd.read_csv("cleaned_100k.csv", low_memory=False)

y = df["Attack Type"]
X = df.drop("Attack Type", axis=1)

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=0.25, random_state=42, stratify=y_train_val)

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)

Train: (60000, 52) Val: (20000, 52) Test: (20000, 52)


### Shared skeleton

In [3]:
# Agreed 10-feature list
FEATURES = [
    "Flow Duration",
    "Total Fwd Packets",
    "Total Length of Fwd Packets",
    "Flow Bytes/s",
    "Flow Packets/s",
    "Fwd Packet Length Mean",
    "Bwd Packet Length Mean",
    "Packet Length Mean",
    "Packet Length Std",
    "Average Packet Size",
]
missing = [f for f in FEATURES if f not in X_train.columns]
assert not missing, f"These features are not in the data: {missing}"

X_train_sel = X_train[FEATURES]
X_val_sel   = X_val[FEATURES]
X_test_sel  = X_test[FEATURES]

CV_FOLDS = 5
SCORING  = "f1_macro"
N_JOBS   = -1

def run_experiment(clf, param_grid, label, use_smote=True, use_pca=True):
    """Fit Pipeline+GridSearchCV and return a results dict."""
    steps = [("scaler", StandardScaler())]
    steps.append(("pca",   PCA() if use_pca else "passthrough"))
    steps.append(("smote", SMOTE(random_state=42) if use_smote else "passthrough"))
    steps.append(("clf",   clf))
    pipe = Pipeline(steps)

    grid = GridSearchCV(pipe, param_grid=param_grid,
                        cv=CV_FOLDS, scoring=SCORING,
                        n_jobs=N_JOBS, verbose=1)
    grid.fit(X_train_sel, y_train)

    y_val_pred  = grid.predict(X_val_sel)
    y_test_pred = grid.predict(X_test_sel)

    val_acc  = accuracy_score(y_val,  y_val_pred)
    val_f1   = f1_score(y_val, y_val_pred, average="macro")
    test_acc = accuracy_score(y_test, y_test_pred)
    test_f1  = f1_score(y_test, y_test_pred, average="macro")

    fig, ax = plt.subplots(figsize=(7, 6))
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_test_pred, ax=ax, xticks_rotation=45, colorbar=False)
    ax.set_title(f"{label} — test confusion matrix")
    plt.tight_layout()
    plt.show()

    print(f"\n=== {label} ===")
    print("Best params :", grid.best_params_)
    print(f"Val  acc={val_acc:.4f}  f1_macro={val_f1:.4f}")
    print(f"Test acc={test_acc:.4f}  f1_macro={test_f1:.4f}")
    print(classification_report(y_test, y_test_pred, digits=3))

    return {
        "label":         label,
        "best_params":   grid.best_params_,
        "val_acc":       val_acc,
        "val_f1_macro":  val_f1,
        "test_acc":      test_acc,
        "test_f1_macro": test_f1,
        "smote":         use_smote,
        "pca":           use_pca,
    }

In [4]:
# Define SVM and parameters
svm_clf = SVC()

svm_param_grid = {
    "pca__n_components": [5, 8],
    "clf__C": [0.1, 1, 10],
    "smote": ["passthrough", SMOTE(random_state=42)]
}

all_results = []

# 1. No SMOTE, PCA
all_results.append(run_experiment(
    svm_clf, svm_param_grid,
    "SVM (no SMOTE, PCA)",
    use_smote=False, use_pca=True))

# 2. No SMOTE, no PCA
all_results.append(run_experiment(
    svm_clf, svm_param_grid,
    "SVM (no SMOTE, no PCA)",
    use_smote=False, use_pca=False))

# 3. SMOTE + PCA
all_results.append(run_experiment(
    svm_clf, svm_param_grid,
    "SVM (SMOTE, PCA)",
    use_smote=True, use_pca=True))

# 4. SMOTE, no PCA
all_results.append(run_experiment(
    svm_clf, svm_param_grid,
    "SVM (SMOTE, no PCA)",
    use_smote=True, use_pca=False))

results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values(by="test_f1", ascending=False)

print("Results:")
print(results_df)

Fitting 5 folds for each of 12 candidates, totalling 60 fits


KeyboardInterrupt: 